# AquaInsight — Task 5: Missing-Data Sensitivity Analysis

**Internship Project:** Predicting Water Quality Index for Comprehensive Water Assessment

This notebook is one of the nine independent GitHub deliverables. It can be run separately using the supplied water-quality CSV.

## Step-by-step approach

1. Load the supplied dataset.
2. Perform the task-specific analysis.
3. Display quantitative results.
4. Interpret the results for downstream water-quality modeling.
5. Preserve data for review rather than making unsupported automatic corrections.

In [ ]:
# Common setup — AquaInsight Water Quality Internship

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

try:
    from rapidfuzz.fuzz import ratio, token_set_ratio
except ImportError:
    raise ImportError("Install RapidFuzz first: pip install rapidfuzz")

possible_paths = [
    Path("Water Quality(1).csv"),
    Path("Water Quality.csv"),
    Path("../data/Water Quality(1).csv"),
    Path("../data/Water Quality.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the supplied CSV beside this notebook.")

df = pd.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head())


In [ ]:
# Build a sample-level water-quality matrix for the task
selected = [
    "pH",
    "Turbidity",
    "Specific conductance",
    "Nitrate",
    "Total Phosphorus, mixed forms",
    "Dissolved oxygen (DO)"
]

wide = (
    df[df["CharacteristicName"].isin(selected)]
    .assign(
        sample_key=lambda x:
            x["MonitoringLocationID"].astype(str) + "_" +
            x["ActivityStartDate"].astype(str)
    )
    .pivot_table(
        index="sample_key",
        columns="CharacteristicName",
        values="ResultValue",
        aggfunc="median"
    )
)

display(wide.head())
print("Wide matrix shape:", wide.shape)


# Task 5 — Missing-Data Sensitivity Analysis

### Internship requirement
Quantify the influence of missing data on Water Quality Index calculations by simulating missingness, evaluating imputation-driven changes, and assessing resilience.

### Steps
1. Convert selected long-form characteristics into a sample-level wide matrix.
2. Profile naturally occurring missingness.
3. Run a compact MCAR-style diagnostic.
4. Simulate 5%, 10%, 20%, and 30% missingness.
5. Impute the simulated missing values.
6. Compare the resulting summary statistics with the original observed values.
7. Interpret the magnitude of change.

### WQI limitation
Because the supplied dataset has no official WQI target/formula, this notebook measures the **impact on water-quality feature summaries**, not an invented WQI score. Once an official WQI formula is provided, the same sensitivity framework should be applied directly to WQI.


In [12]:
# Wide table for a small, interpretable subset of water-quality characteristics
selected = [
    "pH",
    "Turbidity",
    "Specific conductance",
    "Nitrate",
    "Total Phosphorus, mixed forms",
    "Dissolved oxygen (DO)"
]

wide = (
    df[df["CharacteristicName"].isin(selected)]
    .assign(sample_key=lambda x: x["MonitoringLocationID"].astype(str) + "_" + x["ActivityStartDate"].astype(str))
    .pivot_table(index="sample_key", columns="CharacteristicName",
                 values="ResultValue", aggfunc="median")
)

display(wide.head())
print("Wide matrix shape:", wide.shape)

numeric_subset = wide.copy()
print("Missingness (%):")
display((numeric_subset.isna().mean() * 100).round(2).sort_values(ascending=False))

# MCAR-style diagnostic
def little_mcar_test(data):
    data = data.select_dtypes(include="number").dropna(axis=1, how="all")
    data = data.dropna(how="all")

    if data.empty or data.shape[1] == 0:
        return {
            "statistic": np.nan,
            "degrees_of_freedom": 0,
            "p_value": np.nan,
            "interpretation": "No usable numeric data"
        }

    overall_mean = data.mean()
    statistic = 0.0
    degrees_of_freedom = 0

    patterns = data.notna().astype(int).value_counts()

    for pattern, count in patterns.items():
        observed_columns = data.columns[np.asarray(pattern, dtype=bool)]

        if len(observed_columns) == 0:
            continue

        pattern_rows = data.loc[
            data[observed_columns].notna().all(axis=1),
            observed_columns
        ]

        if pattern_rows.empty:
            continue

        pattern_mean = pattern_rows.mean()
        mean_difference = (
            pattern_mean - overall_mean[observed_columns]
        ).to_numpy()

        covariance = data[observed_columns].cov().to_numpy()
        covariance += np.eye(len(observed_columns)) * 1e-8

        statistic += count * (
            mean_difference @ np.linalg.pinv(covariance) @ mean_difference
        )
        degrees_of_freedom += len(observed_columns)

    degrees_of_freedom = max(degrees_of_freedom - data.shape[1], 1)
    p_value = stats.chi2.sf(statistic, degrees_of_freedom)

    return {
        "statistic": statistic,
        "degrees_of_freedom": degrees_of_freedom,
        "p_value": p_value,
        "interpretation": (
            "Evidence against MCAR"
            if p_value < 0.05
            else "No evidence against MCAR"
        )
    }


mcar_input = numeric_subset.sample(min(5000, len(numeric_subset)), random_state=42)
mcar_result = little_mcar_test(mcar_input)
display(pd.DataFrame([mcar_result]))

# Sensitivity experiment
rng = np.random.default_rng(42)
sensitivity_rows = []
for missing_rate in [0.05, 0.10, 0.20, 0.30]:
    observed = numeric_subset.copy()
    mask = rng.random(observed.shape) < missing_rate
    original_mean = observed.mean().mean()
    masked = observed.mask(mask)
    imputed = masked.fillna(masked.median(numeric_only=True))
    imputed_mean = imputed.mean().mean()
    sensitivity_rows.append({
        "simulated_missing_rate": missing_rate,
        "original_mean_of_feature_means": original_mean,
        "imputed_mean_of_feature_means": imputed_mean,
        "absolute_shift": abs(imputed_mean - original_mean)
    })

display(pd.DataFrame(sensitivity_rows).round(4))


CharacteristicName,Dissolved oxygen (DO),Nitrate,Specific conductance,"Total Phosphorus, mixed forms",Turbidity,pH
sample_key,,,,,,
NB01BR0006_2000-09-22,NaN,NaN,NaN,0.032,NaN,NaN
NB01BR0006_2000-10-11,NaN,NaN,NaN,0.038,NaN,NaN
NB01BR0006_2000-11-23,NaN,NaN,NaN,0.011,NaN,NaN
NB01BR0006_2001-04-26,NaN,NaN,27.5,0.024,NaN,4.9
NB01BR0006_2001-05-22,NaN,NaN,23.6,0.015,NaN,5.1


Wide matrix shape: (7274, 6)
Missingness (%):


CharacteristicName
Dissolved oxygen (DO)            75.17
Nitrate                          70.94
Turbidity                        31.08
Total Phosphorus, mixed forms    25.57
Specific conductance              0.71
pH                                0.52
dtype: float64

,statistic,degrees_of_freedom,p_value,interpretation
0,2953.765846,68,0.0,Evidence against MCAR


,simulated_missing_rate,original_mean_of_feature_means,imputed_mean_of_feature_means,absolute_shift
0,0.05,15.7877,15.0547,0.7330
1,0.10,15.7877,14.7530,1.0347
2,0.20,15.7877,13.9216,1.8661
3,0.30,15.7877,13.1316,2.6561


## Task 5 — Conclusion

The analysis above completes the requested **Missing-Data Sensitivity Analysis** component of the AquaInsight internship assignment. Results should be interpreted together with domain requirements and the official WQI definition when it becomes available.